In [9]:
import pandas as pd
import numpy as np
from datetime import datetime

# Read the CSV file
df = pd.read_csv('C:/Users/user/Downloads/vp.csv')

# Convert date strings to datetime objects for easier manipulation
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
df['mat'] = pd.to_datetime(df['mat'], format='%d/%m/%Y')

# Calculate days to maturity for each instrument
df['days_to_maturity'] = (df['mat'] - df['date']).dt.days

# Define target maturity nodes (in days)
target_nodes = [30, 60, 90, 180, 210, 360]

# Initialize list to store results
results = []

# Group by closing run date and interpolate for each date
for date, group in df.groupby('date'):
    # Get days to maturity and rates for this date
    days = group['days_to_maturity'].values
    rates = group['rate'].values
    
    # Sort by days to maturity
    sort_idx = np.argsort(days)
    days_sorted = days[sort_idx]
    rates_sorted = rates[sort_idx]
    
    # Remove any duplicate days (keep first occurrence)
    unique_days, unique_idx = np.unique(days_sorted, return_index=True)
    unique_rates = rates_sorted[unique_idx]
    
    # Interpolate rates at target nodes
    interpolated_rates = np.interp(target_nodes, unique_days, unique_rates)
    
    # Create result row
    row = {'date': date}
    for node, rate in zip(target_nodes, interpolated_rates):
        row[f'{node}d'] = rate
    
    results.append(row)

# Create final DataFrame
df_actual = pd.DataFrame(results)

# Sort by date
df_actual = df_actual.sort_values('date').reset_index(drop=True)

# Insert column with value 4.25 at position 1 (right after the first column)
df_actual.insert(1, '1d', 4.25)

# Convert date back to dd/mm/yyyy string format if needed
# df_constant_maturity['date'] = df_constant_maturity['date'].dt.strftime('%d/%m/%Y')

In [10]:
df_actual.dtypes

date    datetime64[ns]
1d             float64
30d            float64
60d            float64
90d            float64
180d           float64
210d           float64
360d           float64
dtype: object

In [12]:

# Assuming df_actual has columns: 'date', '1', '30d', '60d', '90d', '180d', '210d', '360d'
# Sort by date to ensure proper order
df_actual = df_actual.sort_values('date').reset_index(drop=True)

# Define the maturity nodes (in days)
maturity_nodes = [1, 30, 60, 90, 180, 210, 360]
node_columns = ['1d', '30d', '60d', '90d', '180d', '210d', '360d']

# Initialize forecast dataframe with same structure
df_fcast = df_actual[['date']].copy()
for col in node_columns:
    df_fcast[col] = np.nan

# Calculate forward rates and forecasted curve for each date
for i in range(1, len(df_actual)):
    current_date = df_actual.loc[i, 'date']
    
    # Get previous day's curve
    prev_curve = df_actual.loc[i-1, node_columns].values.astype(float)
    
    # Calculate number of days between t-1 and t
    days_elapsed = (current_date - df_actual.loc[i-1, 'date']).days
    
    # Calculate forward rates for each node from previous curve
    forward_rates = []
    forward_days = []
    
    for j, (maturity, rate) in enumerate(zip(maturity_nodes, prev_curve)):
        if maturity == 1:
            # 1d rate remains at 4.25
            forward_rates.append(4.25)
            forward_days.append(1)
        else:
            # Calculate forward rate from day 'days_elapsed' to 'maturity'
            rate_long = rate / 100  # Convert to decimal
            rate_short = prev_curve[0] / 100  # 1d rate from previous day (4.25%)
            
            t_long = maturity / 360
            t_short = days_elapsed / 360
            t_forward = (maturity - days_elapsed) / 360
            
            # Forward rate formula: ((1 + r_long)^t_long / (1 + r_short)^t_short)^(1/t_forward) - 1
            fwd_rate = ((1 + rate_long) ** t_long / (1 + rate_short) ** t_short) ** (1 / t_forward) - 1
            fwd_rate_annual = fwd_rate * 100  # Convert back to percentage
            
            forward_rates.append(fwd_rate_annual)
            forward_days.append(maturity - days_elapsed)
    
    # Sort by maturity days
    sort_idx = np.argsort(forward_days)
    forward_days_sorted = np.array(forward_days)[sort_idx]
    forward_rates_sorted = np.array(forward_rates)[sort_idx]
    
    # Remove any duplicate or negative maturities
    valid_idx = forward_days_sorted > 0
    forward_days_sorted = forward_days_sorted[valid_idx]
    forward_rates_sorted = forward_rates_sorted[valid_idx]
    
    # Interpolate to get forecasted rates at target nodes
    forecasted_rates = np.interp(maturity_nodes, forward_days_sorted, forward_rates_sorted)
    
    # Ensure 1d rate is exactly 4.25
    forecasted_rates[0] = 4.25
    
    # Store forecasted rates
    for k, col in enumerate(node_columns):
        df_fcast.loc[i, col] = forecasted_rates[k]

# First date has no forecast (no previous day)
# df_fcast.loc[0, node_columns] remains as NaN